# Bike Sharing 线性回归
本 Notebook 只读取仓库内已验证的 17,379 行快照，并使用固定时间顺序划分。

In [1]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

FEATURE_ORDER = ("temp", "hum", "windspeed", "workingday", "hr")
CONTINUOUS_FEATURES = ("temp", "hum", "windspeed", "hr")
SPLIT_INDEX = 13_903
SOURCE_SHA256 = "e03de4ee4ef4dc376ac6e04bf829673c6269e8eba5c60fa121640fa2f829504f"
METHOD_TOLERANCE = 1e-6
LEARNING_RATE = 0.1
MAX_UPDATES = 5_000
GRADIENT_TOLERANCE = 1e-8
DATASET_PATH = Path("../../datasets/python-data-tools/bike-sharing-hour.csv")

## 同一数据、同一特征顺序
主设计矩阵固定为 temp、hum、windspeed、workingday、hr。casual 与 registered 因为满足 casual + registered = cnt，必须排除。

In [2]:
assert hashlib.sha256(DATASET_PATH.read_bytes()).hexdigest() == SOURCE_SHA256
frame = pd.read_csv(DATASET_PATH)
assert len(frame) == 17_379
assert tuple(frame.columns) == (
    "instant", "dteday", "season", "yr", "mnth", "hr", "holiday",
    "weekday", "workingday", "weathersit", "temp", "atemp", "hum",
    "windspeed", "casual", "registered", "cnt",
)
assert np.array_equal(frame["instant"].to_numpy(), np.arange(1, 17_380))
assert np.array_equal(frame["casual"] + frame["registered"], frame["cnt"])
print(json.dumps({
    "sourceSha256": SOURCE_SHA256,
    "rows": len(frame),
    "target": "cnt",
    "leakageExcluded": ["casual", "registered"],
}, sort_keys=True))

{"leakageExcluded": ["casual", "registered"], "rows": 17379, "sourceSha256": "e03de4ee4ef4dc376ac6e04bf829673c6269e8eba5c60fa121640fa2f829504f", "target": "cnt"}


In [3]:
train = frame.iloc[:SPLIT_INDEX].copy()
held_out = frame.iloc[SPLIT_INDEX:].copy()
assert int(train.iloc[-1]["instant"]) == 13_903
assert int(held_out.iloc[0]["instant"]) == 13_904
scaler = StandardScaler()
X_train_continuous = scaler.fit_transform(train.loc[:, list(CONTINUOUS_FEATURES)])
X_held_out_continuous = scaler.transform(held_out.loc[:, list(CONTINUOUS_FEATURES)])

def build_matrix(partition, scaled):
    scaled_columns = {
        feature: scaled[:, index]
        for index, feature in enumerate(CONTINUOUS_FEATURES)
    }
    return np.column_stack([
        partition["workingday"].to_numpy(float)
        if feature == "workingday"
        else scaled_columns[feature]
        for feature in FEATURE_ORDER
    ])

X_train = build_matrix(train, X_train_continuous)
X_held_out = build_matrix(held_out, X_held_out_continuous)
y_train = train["cnt"].to_numpy(float)
y_held_out = held_out["cnt"].to_numpy(float)
assert np.array_equal(X_train[:, 3], train["workingday"].to_numpy(float))
scaler_table = pd.DataFrame({
    "feature": CONTINUOUS_FEATURES,
    "train_mean": scaler.mean_,
    "train_population_scale": scaler.scale_,
})
print(scaler_table.to_string(index=False))

  feature  train_mean  train_population_scale
     temp    0.499170                0.197709
      hum    0.622996                0.198187
windspeed    0.194097                0.123019
       hr   11.546573                6.911987


## 正规方程与稳定实现
概念映射为 `X_tilde = [1, X]`、`theta = (X_tilde^T X_tilde)^+ X_tilde^T y`（也可简写为 `theta = pinv(X_tilde) @ y`）、`theta[0] = b`、`theta[1:] = w`。代码使用 `numpy.linalg.lstsq`，避免显式求逆并保留秩与奇异值诊断。三种方法的角色分别是 NumPy batch gradient descent、正规方程数值参考和 scikit-learn LinearRegression。

In [4]:
def gradient_state(matrix, target, weights, intercept):
    residual = matrix @ weights + intercept - target
    mse = float(np.mean(residual * residual))
    weight_gradient = (2.0 / len(matrix)) * (matrix.T @ residual)
    intercept_gradient = float(2.0 * np.mean(residual))
    gradient_norm = float(np.linalg.norm(
        np.append(weight_gradient, intercept_gradient)
    ))
    assert np.isfinite(np.append(
        [mse, intercept_gradient, gradient_norm],
        weight_gradient,
    )).all()
    return mse, weight_gradient, intercept_gradient, gradient_norm

gd_weights = np.zeros(X_train.shape[1], dtype=float)
gd_intercept = 0.0
trace_rows = []
for update in range(MAX_UPDATES + 1):
    mse, weight_gradient, intercept_gradient, gradient_norm = gradient_state(
        X_train, y_train, gd_weights, gd_intercept
    )
    trace_rows.append({
        "update": update,
        "mse": mse,
        "gradient_norm": gradient_norm,
        "intercept": gd_intercept,
        **dict(zip(FEATURE_ORDER, gd_weights, strict=True)),
    })
    if gradient_norm <= GRADIENT_TOLERANCE:
        break
    assert update < MAX_UPDATES
    gd_weights = gd_weights - LEARNING_RATE * weight_gradient
    gd_intercept = float(gd_intercept - LEARNING_RATE * intercept_gradient)

gd_trace = pd.DataFrame(trace_rows)
assert update == 772
assert gradient_norm <= GRADIENT_TOLERANCE
print(json.dumps({
    "updates": update,
    "reason": "gradient-tolerance",
    "gradientNorm": gradient_norm,
    "learningRate": LEARNING_RATE,
}, sort_keys=True))
print(gd_trace.to_csv(index=False, float_format="%.17g"))

{"gradientNorm": 9.964423234025087e-09, "learningRate": 0.1, "reason": "gradient-tolerance", "updates": 772}
update,mse,gradient_norm,intercept,temp,hum,windspeed,workingday,hr
0,58370.935337696901,482.15589091496292,0,0,0,0,0,0
1,38620.66078664668,337.37434399142529,34.927828526217368,14.386783408580655,-10.967267693811927,3.3851847369039016,24.235129108825433,13.368933006998928
2,28936.042444480514,237.03390983556369,59.552512571393478,25.279224772221539,-18.658855659366488,5.1443026932203564,40.324151395895292,22.964894505536257
3,24144.663234518972,167.55447989087895,77.049812334560812,33.555700841602118,-24.063787521710307,5.8835319338499525,50.802925023010253,29.866397526985665
4,21742.262262607135,119.54511389728471,89.613199007849104,39.866578674459888,-27.871722071082914,6.0038767845510961,57.423157144933555,34.840013095932989
5,20512.842804822532,86.511784573299394,98.75765604306001,44.695232677088832,-30.563470213491264,5.767374990305461,61.394725307287651,38.43146876072408


In [5]:
# Conceptual normal equation / 正规方程:
# X_tilde = [1, X]
# theta = (X_tilde^T X_tilde)^+ X_tilde^T y
# The stable executable authority is numpy.linalg.lstsq, not an explicit inverse.
X_tilde = np.column_stack([np.ones(len(X_train)), X_train])
theta, residual_sums, rank, singular_values = np.linalg.lstsq(
    X_tilde, y_train, rcond=None
)
b = theta[0]
w = theta[1:]
sklearn_model = LinearRegression(fit_intercept=True).fit(X_train, y_train)
lstsq_train_prediction = X_train @ w + b
lstsq_test_prediction = X_held_out @ w + b
gd_test_prediction = X_held_out @ gd_weights + gd_intercept
sklearn_test_prediction = sklearn_model.predict(X_held_out)
np.testing.assert_allclose(gd_weights, w, rtol=0.0, atol=METHOD_TOLERANCE)
np.testing.assert_allclose(gd_intercept, b, rtol=0.0, atol=METHOD_TOLERANCE)
np.testing.assert_allclose(
    sklearn_model.coef_, w, rtol=0.0, atol=METHOD_TOLERANCE
)
np.testing.assert_allclose(
    sklearn_model.intercept_, b, rtol=0.0, atol=METHOD_TOLERANCE
)
np.testing.assert_allclose(
    gd_test_prediction, lstsq_test_prediction, rtol=0.0, atol=METHOD_TOLERANCE
)
np.testing.assert_allclose(
    sklearn_test_prediction,
    lstsq_test_prediction,
    rtol=0.0,
    atol=METHOD_TOLERANCE,
)

def metrics(actual, prediction):
    return {
        "mse": float(mean_squared_error(actual, prediction)),
        "mae": float(mean_absolute_error(actual, prediction)),
        "r2": float(r2_score(actual, prediction)),
    }

method_table = pd.DataFrame([
    {"method": method, "feature": feature, "coefficient": float(value)}
    for method, weights, intercept in (
        ("numpy-batch-gradient-descent", gd_weights, gd_intercept),
        ("numpy-lstsq", w, b),
        ("sklearn-linear-regression", sklearn_model.coef_, sklearn_model.intercept_),
    )
    for feature, value in (
        ("intercept", intercept),
        *zip(FEATURE_ORDER, weights, strict=True),
    )
])
method_delta = {
    "tolerance": METHOD_TOLERANCE,
    "maxCoefficientDelta": float(max(
        np.max(np.abs(np.append(gd_intercept, gd_weights) - theta)),
        np.max(np.abs(
            np.append(sklearn_model.intercept_, sklearn_model.coef_) - theta
        )),
    )),
    "maxPredictionDelta": float(max(
        np.max(np.abs(gd_test_prediction - lstsq_test_prediction)),
        np.max(np.abs(sklearn_test_prediction - lstsq_test_prediction)),
    )),
}
metric_output = {
    "train": metrics(y_train, lstsq_train_prediction),
    "test": metrics(y_held_out, lstsq_test_prediction),
    "normalEquation": {
        "term": {"en": "normal equation", "zh-CN": "正规方程"},
        "augmentedDesign": "X_tilde = [1, X]",
        "formula": "theta = (X_tilde^T X_tilde)^+ X_tilde^T y",
        "interceptMapping": "theta[0] = b",
        "weightMapping": "theta[1:] = w",
        "implementation": "numpy.linalg.lstsq",
        "rank": int(rank),
        "singularValues": [float(value) for value in singular_values],
        "conditionNumber": float(np.linalg.cond(X_tilde)),
    },
    "methodDelta": method_delta,
}
print(method_table.to_string(index=False))
print(json.dumps(metric_output, sort_keys=True))

                      method    feature  coefficient
numpy-batch-gradient-descent  intercept   173.010328
numpy-batch-gradient-descent       temp    62.723891
numpy-batch-gradient-descent        hum   -37.116416
numpy-batch-gradient-descent  windspeed     0.809446
numpy-batch-gradient-descent workingday     2.379719
numpy-batch-gradient-descent         hr    47.901434
                 numpy-lstsq  intercept   173.010328
                 numpy-lstsq       temp    62.723891
                 numpy-lstsq        hum   -37.116416
                 numpy-lstsq  windspeed     0.809446
                 numpy-lstsq workingday     2.379719
                 numpy-lstsq         hr    47.901434
   sklearn-linear-regression  intercept   173.010328
   sklearn-linear-regression       temp    62.723891
   sklearn-linear-regression        hum   -37.116416
   sklearn-linear-regression  windspeed     0.809446
   sklearn-linear-regression workingday     2.379719
   sklearn-linear-regression         hr    47.

## 可复核记录
普通训练行、负预测、早高峰低估、晚高峰低估和大残差记录都由固定筛选规则与最低 instant 并列规则确定；完整计算与输出由 Plan 27-03 生成。

In [6]:
train_residual = lstsq_train_prediction - y_train
held_residual = lstsq_test_prediction - y_held_out
q1, q3 = np.quantile(y_train, [0.25, 0.75])
eligible = np.flatnonzero((y_train >= q1) & (y_train <= q3))
representative_position = min(
    eligible,
    key=lambda position: (
        abs(float(train_residual[position])),
        int(train.iloc[position]["instant"]),
    ),
)
negative_position = min(
    range(len(held_out)),
    key=lambda position: (
        float(lstsq_test_prediction[position]),
        int(held_out.iloc[position]["instant"]),
    ),
)
underprediction = y_held_out - lstsq_test_prediction
hours = held_out["hr"].to_numpy(int)
morning_position = min(
    np.flatnonzero((hours >= 7) & (hours <= 9) & (underprediction > 0)),
    key=lambda position: (
        -float(underprediction[position]),
        int(held_out.iloc[position]["instant"]),
    ),
)
evening_position = min(
    np.flatnonzero((hours >= 16) & (hours <= 19) & (underprediction > 0)),
    key=lambda position: (
        -float(underprediction[position]),
        int(held_out.iloc[position]["instant"]),
    ),
)
excluded = {negative_position, morning_position, evening_position}
large_position = min(
    (position for position in range(len(held_out)) if position not in excluded),
    key=lambda position: (
        -abs(float(held_residual[position])),
        int(held_out.iloc[position]["instant"]),
    ),
)
role_positions = [
    ("negative-prediction", negative_position),
    ("morning-peak-underprediction", int(morning_position)),
    ("evening-peak-underprediction", int(evening_position)),
    ("large-residual", large_position),
]
named_cases = [{
    "role": role,
    "instant": int(held_out.iloc[position]["instant"]),
    "timestamp": (
        f"{held_out.iloc[position]['dteday']} "
        f"{int(held_out.iloc[position]['hr']):02d}:00"
    ),
    "actual": float(y_held_out[position]),
    "prediction": float(lstsq_test_prediction[position]),
    "residual": float(held_residual[position]),
} for role, position in role_positions]
resolved_instants = [
    int(train.iloc[representative_position]["instant"]),
    *[row["instant"] for row in named_cases],
]
assert resolved_instants == [11_550, 17_213, 15_628, 14_965, 15_604]

residual_frame = pd.DataFrame({
    "hour": hours,
    "prediction": lstsq_test_prediction,
    "residual": held_residual,
    "absolute_residual": np.abs(held_residual),
})
hourly_residual = (
    residual_frame.groupby("hour", sort=True)["residual"].mean().reset_index()
)
residual_frame["prediction_bin"], bin_edges = pd.qcut(
    residual_frame["prediction"],
    q=4,
    labels=False,
    retbins=True,
    duplicates="raise",
)
prediction_bins = residual_frame.groupby("prediction_bin", sort=True).agg(
    residual_std_dev=("residual", lambda values: float(np.std(values, ddof=0))),
    mae=("absolute_residual", "mean"),
    rows=("residual", "size"),
).reset_index()

extended_continuous = ("temp", "atemp", "hum", "windspeed", "hr")
extended_scaler = StandardScaler().fit(train.loc[:, list(extended_continuous)])
extended_train_scaled = extended_scaler.transform(
    train.loc[:, list(extended_continuous)]
)
extended_test_scaled = extended_scaler.transform(
    held_out.loc[:, list(extended_continuous)]
)
extended_train = np.column_stack([
    extended_train_scaled[:, 0],
    extended_train_scaled[:, 1],
    extended_train_scaled[:, 2],
    extended_train_scaled[:, 3],
    train["workingday"].to_numpy(float),
    extended_train_scaled[:, 4],
])
extended_test = np.column_stack([
    extended_test_scaled[:, 0],
    extended_test_scaled[:, 1],
    extended_test_scaled[:, 2],
    extended_test_scaled[:, 3],
    held_out["workingday"].to_numpy(float),
    extended_test_scaled[:, 4],
])
atemp_ols = LinearRegression().fit(extended_train, y_train)
ridge = Ridge(alpha=300.0).fit(extended_train, y_train)
lasso = Lasso(
    alpha=0.1, max_iter=100_000, tol=1e-10, selection="cyclic"
).fit(extended_train, y_train)
log_model = LinearRegression().fit(X_train, np.log1p(y_train))
log_count_prediction = np.expm1(log_model.predict(X_held_out))
diagnostics = {
    "resolvedInstants": resolved_instants,
    "namedCases": named_cases,
    "hourlyResiduals": hourly_residual.to_dict(orient="records"),
    "predictionBins": prediction_bins.to_dict(orient="records"),
    "collinearity": {
        "addedFeature": "atemp",
        "tempAtempTrainingCorrelation": float(np.corrcoef(
            train["temp"], train["atemp"]
        )[0, 1]),
        "conditionNumber": float(np.linalg.cond(np.column_stack([
            np.ones(len(extended_train)), extended_train
        ]))),
        "olsTemp": float(atemp_ols.coef_[0]),
        "olsAtemp": float(atemp_ols.coef_[1]),
        "olsTestMetrics": metrics(y_held_out, atemp_ols.predict(extended_test)),
        "ridgeObjective": "mse-plus-l2",
        "ridgeAlpha": 300.0,
        "ridgeTestMetrics": metrics(y_held_out, ridge.predict(extended_test)),
        "lassoObjective": "mse-plus-l1",
        "lassoAlpha": 0.1,
        "lassoTestMetrics": metrics(y_held_out, lasso.predict(extended_test)),
    },
    "log1p": {
        "rawTargetObjectiveComparable": False,
        "inverseTransform": "expm1",
        "countScaleMetrics": metrics(y_held_out, log_count_prediction),
    },
}
print(hourly_residual.to_string(index=False))
print(prediction_bins.to_string(index=False))
print(pd.DataFrame(named_cases).to_string(index=False))
print(json.dumps(diagnostics, sort_keys=True))

 hour    residual
    0   -3.246337
    1   22.936305
    2   41.946037
    3   59.661590
    4   68.676896
    5   54.255679
    6  -13.214120
    7 -187.983968
    8 -367.418651
    9 -151.626925
   10  -68.780154
   11  -95.365229
   12 -140.838565
   13 -121.258960
   14  -90.131868
   15 -102.174050
   16 -181.233505
   17 -366.629323
   18 -309.913374
   19 -159.439159
   20  -49.898556
   21   19.939029
   22   70.748656
   23  118.141987
 prediction_bin  residual_std_dev        mae  rows
              0        135.914828  78.378245   869
              1        185.603132 133.925750   869
              2        176.649153 147.177245   869
              3        209.112513 181.705322   869
                        role  instant        timestamp  actual  prediction    residual
         negative-prediction    17213 2012-12-25 00:00    13.0  -47.415493  -60.415493
morning-peak-underprediction    15628 2012-10-18 08:00   834.0  101.882097 -732.117903
evening-peak-underprediction    14

## 完整结果与限制
下方共享代码给出完整 GD 轨迹、三方法系数和容差、训练/留出指标、小时残差、预测分箱、五个确定记录、仅新增 atemp 的共线性对照，以及不同目标尺度的 log1p 对照。方法一致只说明优化完成，不代表线性模型已经充分。

In [7]:
assert method_delta["maxCoefficientDelta"] <= METHOD_TOLERANCE
assert method_delta["maxPredictionDelta"] <= METHOD_TOLERANCE
assert len(gd_trace) == 773
assert len(held_out) == 3_476
assert np.isfinite(method_table["coefficient"]).all()
assert np.isfinite(gd_trace.select_dtypes(include=[np.number])).all().all()
assert np.isfinite(held_residual).all()
print(json.dumps({
    "assertionsPassed": True,
    "codeAuthority": "shared-byte-identical-blueprint",
    "normalEquationImplementation": "numpy.linalg.lstsq",
    "residualSign": "prediction - actual",
    "completeTraceRows": len(gd_trace),
    "completeHeldoutResidualRows": len(held_out),
    "resolvedInstants": resolved_instants,
}, sort_keys=True))

{"assertionsPassed": true, "codeAuthority": "shared-byte-identical-blueprint", "completeHeldoutResidualRows": 3476, "completeTraceRows": 773, "normalEquationImplementation": "numpy.linalg.lstsq", "residualSign": "prediction - actual", "resolvedInstants": [11550, 17213, 15628, 14965, 15604]}
